# AVISO Eddy Dataset Pipeline

This notebook is a concise control panel for the AVISO surface eddy pipeline. Scientific and orchestration code lives in `src/aviso_eddy_dataset/`; this notebook loads configuration, runs stages, and checks outputs.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

AVISO_SRC = PROJECT_ROOT / "src"
SEACOFS_SRC = PROJECT_ROOT.parent / "seacofs_eddy_dataset_modular" / "src"
for source in (AVISO_SRC, SEACOFS_SRC):
    if str(source) not in sys.path:
        sys.path.insert(0, str(source))

PROJECT_ROOT

## Load configuration

The notebook uses `config/local.yaml` when present and otherwise falls back to the committed `config/example.yaml`.

In [ ]:
from aviso_eddy_dataset.config import load_config
from aviso_eddy_dataset.pipeline import STAGES, run_all, run_stage

LOCAL_CONFIG = PROJECT_ROOT / "config" / "local.yaml"
CONFIG_PATH = LOCAL_CONFIG if LOCAL_CONFIG.exists() else PROJECT_ROOT / "config" / "example.yaml"
config = load_config(CONFIG_PATH)

print(f"Config: {CONFIG_PATH}")
print(f"Input:  {config.data_root}")
print(f"Output: {config.output_root}")

In [ ]:
[(stage.name, stage.description) for stage in STAGES]

## Run one stage

Uncomment a stage while developing or resuming the workflow.

In [ ]:
# run_stage("detect_nencioli", config)
# run_stage("fit_doppio_surface", config)
# run_stage("track_eddies", config)
# run_stage("process_tracked_dataset", config)

## Run selected stages

Uncomment the stages to run. Keep their pipeline order. With `skip_existing: true`, completed annual partitions are skipped.

In [ ]:
STAGES_TO_RUN = [
    # "detect_nencioli",
    # "fit_doppio_surface",
    # "track_eddies",
    # "process_tracked_dataset",
]

for stage_name in STAGES_TO_RUN:
    run_stage(stage_name, config)

## Inspect outputs

In [ ]:
import pandas as pd

output_dirs = {
    "detections": config.output_root / "detections",
    "surface_eddies": config.output_root / "surface_eddies",
    "tracked": config.output_root / "tracked",
    "processed": config.output_root / "processed",
}

pd.DataFrame(
    [
        {
            "stage": name,
            "path": str(path),
            "parquet_files": len(list(path.glob("*.parquet"))) if path.exists() else 0,
        }
        for name, path in output_dirs.items()
    ]
)

In [ ]:
detection_files = sorted((config.output_root / "detections").glob("source=*.parquet"))

if detection_files:
    detection_counts = []
    for path in detection_files:
        frame = pd.read_parquet(path, columns=["Day", "Cyc"])
        detection_counts.append(
            {
                "partition": path.stem,
                "rows": len(frame),
                "first_day": frame.Day.min() if not frame.empty else pd.NA,
                "last_day": frame.Day.max() if not frame.empty else pd.NA,
                "AE": int(frame.Cyc.eq("AE").sum()),
                "CE": int(frame.Cyc.eq("CE").sum()),
            }
        )
    display(pd.DataFrame(detection_counts))
else:
    print(f"No detection files found in {config.output_root / 'detections'}")

In [ ]:
for name, path in {
    "tracked": config.output_root / "tracked" / "eddy_tracks.parquet",
    "processed": config.output_root / "processed" / "eddy_dataset_processed.parquet",
}.items():
    if path.exists():
        frame = pd.read_parquet(path)
        print(f"{name}: {len(frame):,} rows, {frame['Eddy'].nunique() if 'Eddy' in frame else 0:,} eddies")
        display(frame.head())
    else:
        print(f"Missing {name} output: {path}")

## Run everything

This executes all four stages in order.

In [ ]:
# run_all(config)